## Dataset Setup

In [ ]:
from pathlib import Path
import os

AIST_DIR = Path("/content/datasets/AIST/")
AIST_DIR.mkdir(parents=True, exist_ok=True)

AlphaPose_DIR = Path("/content/preprocessors/AlphaPose")
if not os.path.exists(AlphaPose_DIR):
    !git clone -q https://github.com/MVIG-SJTU/AlphaPose.git {AlphaPose_DIR}
    print(f"AlphaPose cloned @{AlphaPose_DIR}")
else:
    print(f"Skipping AlphaPose already exists @{AlphaPose_DIR}")

Skipping AlphaPose already exists @/content/preprocessors/AlphaPose


In [ ]:
# Download SMPL models from Google Drive
import gdown
from pathlib import Path

SMPL_DIR = Path("/content/datasets/SMPL/")
SMPL_DIR.mkdir(parents=True, exist_ok=True)

# Link: https://drive.google.com/drive/folders/15LENU5oO56TYcWz0sEsWEUsUjCFKJz7s?usp=sharing
# For now, trying with the folder ID directly.
drive_folder_id = "15LENU5oO56TYcWz0sEsWEUsUjCFKJz7s"
output_path = str(SMPL_DIR)

print(f"Attempting to download SMPL models to {SMPL_DIR}...")

# Using gdown to download the folder content.
# Note: gdown's folder download capability can be inconsistent.
!gdown --folder "{drive_folder_id}" --output "{output_path}" --no-verbose --fuzzy

print("Download attempt finished.")

### Refernces:
1. [Data Formats](https://google.github.io/aistplusplus_dataset/download.html)
2. [SMPL Format](https://medium.com/@bhipanshudhupar/what-is-smpl-the-3d-human-body-model-powering-modern-ai-and-animation-c654a0284800)
3. [SMPL made simple FAQs](https://files.is.tue.mpg.de/black/talks/SMPL-made-simple-FAQs.pdf)

FYI:
- **motions** store SMPL parameters only (refer 6D rotational representation in paper),i.e., R^{S×(24×6+4+3)} where J3D=24 (each 6 param), foot contact labels = R^4, root pos = R^3.

- **keypoints2d:** Multi-view frame-by-frame 2D keypoints detection results. Array shape is (9, N, 17, 3) where
    - The first dim represents Individual environment settings, each with 9 cameras.
    - J2D = 17 (no.of joints in 2d repr. w.r.t coco semantics).
    - The last dim, i.e. each joint contains (x, y, confidence).
    
- **keypoints3d:** Raw reconstructed 3D joint coordinates (i.e. last dim is 3D: x,y,z) and smoothed/optimized versions (keypoints3d_optim) plus metadata. Array shape is (N, 17, 3).

- **Timestamps:** Annotations are frame-by-frame under exact 60 FPS. Some videos in the AIST Dance Video DB have slightly different FPS, but hard-coded 60 FPS when converting videos into images.

- Canonical **SMPL joint id** → name mapping used by SMPL/SMPL-X:
| | | |
|---|---|---|
| 0:  Pelvis    | 1:  L_Hip     | 2:  R_Hip     |
| 3:  Spine1    | 4:  L_Knee    | 5:  R_Knee    |
| 6:  Spine2    | 7:  L_Ankle   | 8:  R_Ankle   |
| 9:  Spine3    | 10: L_Foot    | 11: R_Foot    |
| 12: Neck      | 13: L_Collar  | 14: R_Collar  |
| 15: Head      | 16: L_Shoulder| 17: R_Shoulder|
|18: L_Elbow, 19: R_Elbow | 20: L_Wrist, 21: R_Wrist |22: L_Hand, 23: R_Hand |


In [ ]:
mocap_zip = AIST_DIR / "motions.zip"
mocap_dir = mocap_zip.with_suffix('')
keypoints_zip = AIST_DIR / "keypoints2d.zip"
keypoints_dir = keypoints_zip.with_suffix('')

# Check and download motions.zip
if not mocap_dir.exists():
    if not mocap_zip.exists():
        print(f"File {mocap_zip.name} does not exist. Downloading...")
        !wget -c -q https://storage.googleapis.com/aist_plusplus_public/20210308/motions.zip -O {mocap_zip} \
        || echo "Failed to download {mocap_zip}!"
    else:
        print(f"File {mocap_zip.name} already exists. Skipping download.")
    !unzip -q {mocap_zip} -d {AIST_DIR} || echo "{mocap_zip} unzip failed"
print(f"Download & Unzip checks completed in '{mocap_dir}' :)")

# Check and download keypoints2d.zip
if not keypoints_dir.exists():
    if not keypoints_zip.exists():
        print(f"File {keypoints_zip.name} does not exist. Downloading...")
        !wget -c -q https://storage.googleapis.com/aist_plusplus_public/20210308/keypoints2d.zip -O {keypoints_zip} \
        || echo "Failed to download {keypoints_zip}!"
    else:
        print(f"File {keypoints_zip.name} already exists. Skipping download.")
    !unzip -q {keypoints_zip} -d {AIST_DIR} || echo "{keypoints_zip} unzip failed"
print(f"Download & Unzip checks completed in '{keypoints_dir}' :)")

File motions.zip already exists. Skipping download.
Download & Unzip checks completed in /content/datasets/AIST/motions :)
File keypoints2d.zip already exists. Skipping download.
Download & Unzip checks completed in /content/datasets/AIST/keypoints2d :)


In [ ]:
# quick test: open one motion pkl and one keypoints2d pkl to verify fields
import pickle, glob, os
M3D_pkl = glob.glob(f"{mocap_dir}/*.pkl")
P3D_pkl = glob.glob(f"{keypoints_dir}/*.pkl")

print("Found motion files:", len(M3D_pkl), "Found keypoints2d files:", len(P3D_pkl))

if M3D_pkl:
    with open(M3D_pkl[0], "rb") as f:
        m = pickle.load(f)
    print("\nMotion keys:", list(m.keys()))
    for k,v in m.items():
        print(f"{k} shape:", v.shape if hasattr(v, 'shape') else type(v), end=", ")
    print()


if P3D_pkl:
    with open(P3D_pkl[0], "rb") as f:
        p = pickle.load(f)
    print("\nKeypoints keys:", list(p.keys()))
    for k,v in p.items():
        print(f"{k} shape:", v.shape if hasattr(v, 'shape') else type(v), end=", ")
    print()


Found motion files: 1408 Found keypoints2d files: 1510

Motion keys: ['smpl_loss', 'smpl_poses', 'smpl_scaling', 'smpl_trans']
smpl_loss shape: <class 'float'>, smpl_poses shape: (480, 72), smpl_scaling shape: (1,), smpl_trans shape: (480, 3), 

Keypoints keys: ['keypoints2d', 'det_scores', 'timestamps']
keypoints2d shape: (9, 480, 17, 3), det_scores shape: (9, 480), timestamps shape: (480,), 


Note: each data-entry: no.of time-stamps: 480 => 8 secs of 60 fps motions

### Preprocess & pack dataset for ViMo training
- Resamples 60 → 30 FPS by taking every 2nd frame (paper re-aligns to 30 FPS for training).
- Cuts sequences into 5s clips (S = 150 frames).
- Keeps cond_2d as views x 150 x 17 x 3 (multi-view 2D detections).
- Builds target_m per clip as 150 x 151 (24×6 rotations + 4 foot contact + 3 root).
- Skips sequences listed in ignore_list.txt if present.

In [ ]:
# Helper: axis-angle (24x3) -> rotation matrices -> 6D
def axisangle_to_rotmat_batch(axisangle):
    # axisangle: (S,24,3) or (N,24,3)
    # returns R: (S,24,3,3)
    axisangle = np.asarray(axisangle)
    if axisangle.ndim == 2:
        axisangle = axisangle.reshape(axisangle.shape[0]//24, 24, 3)
    Sx = axisangle.shape[0]
    R = np.zeros((Sx, 24, 3, 3), dtype=np.float32)
    for t in range(Sx):
        for j in range(24):
            v = axisangle[t,j]
            theta = np.linalg.norm(v)
            if theta < 1e-8:
                R[t,j] = np.eye(3, dtype=np.float32)
            else:
                k = (v / theta).astype(np.float32)
                K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]], dtype=np.float32)
                R[t,j] = np.eye(3, dtype=np.float32) + np.sin(theta)*K + (1-np.cos(theta))*(K @ K)
    return R

def rotmat_to_6d(R):
    # R: (...,3,3) -> take first two cols -> (...,6)
    cols = R[..., :2]  # (...,3,2)
    return cols.reshape(*R.shape[:-2], 6)

def compute_foot_contact_from_joints(joint_pos, foot_idx=(11,14), toe_idx=(16,17), vel_thresh=1e-3):
    # joint_pos: S x J x 3 (world positions). AIST motions may not provide joint world positions directly.
    # We'll compute foot contact via vertical velocity of foot and toe joints if available.
    S = joint_pos.shape[0]
    contacts = np.zeros((S,4), dtype=np.int8)
    vel = np.zeros_like(joint_pos)
    vel[1:] = joint_pos[1:] - joint_pos[:-1]
    # up-axis in SMPL coordinate is y (paper checks up-axis)
    for t in range(1,S):
        # foot pair
        contacts[t,0] = int(np.all(np.abs(vel[t,foot_idx[0],1]) < vel_thresh))
        contacts[t,1] = int(np.all(np.abs(vel[t,foot_idx[1],1]) < vel_thresh))
        contacts[t,2] = int(np.all(np.abs(vel[t,toe_idx[0],1]) < vel_thresh))
        contacts[t,3] = int(np.all(np.abs(vel[t,toe_idx[1],1]) < vel_thresh))
    return contacts

In [ ]:
# convert AIST++ annotations -> ViMo training samples (.npz)
import os, glob, pickle, numpy as np
from pathlib import Path
from tqdm import tqdm

OUT_DIR = AIST_DIR / "processed"   # output ViMo-ready dataset
OUT_DIR.mkdir(parents=True, exist_ok=True)

S = 150   # frames per clip (5s * 30 fps)
SRC_FPS = 60
TARGET_FPS = 30
DOWNSAMPLE = SRC_FPS // TARGET_FPS  # should be 2

# iterate sequences, pair motion and keypoints by filename prefix
count_saved = 0
for m in tqdm(M3D_pkl):
    name = Path(m).stem
    p = os.path.join(keypoints_dir, name + ".pkl")
    if not os.path.exists(p): # skip if not found
        continue

    pdata = pickle.load(open(p, "rb"))
    p2d_all = pdata.get("keypoints2d", None)
    if p2d_all is None:
        continue
    mdata = pickle.load(open(m, "rb"))
    poses = mdata.get("smpl_poses", None)   # axis-angle per joint
    trans = mdata.get("smpl_trans", None)   # root translation or global position
    if poses is None or trans is None:
        continue

    # downsample from 60->30 fps by selecting every DOWNSAMPLE-th frame
    p2d_ds = p2d_all[:, ::DOWNSAMPLE]   # shape (views, N_ds, 17, 3)
    poses_ds = poses[::DOWNSAMPLE]  # (N_ds,24,3)
    trans_ds = trans[::DOWNSAMPLE]  # (N_ds,3)
    N_ds = poses_ds.shape[0]

    # cut into S-length clips (pad last clip with zeros if shorter)
    num_clips = int(np.ceil(N_ds / S))
    for i in range(num_clips):
        st = i*S
        ed = min((i+1)*S, N_ds)
        # cond_2d: keep all views (9) as paper used multi-view projected 2D
        clip_p2d = p2d_ds[:, st:ed]   # (views, tlen, 17, 3)
        # pad if needed
        if clip_p2d.shape[1] < S:
            pad_len = S - clip_p2d.shape[1]
            clip_p2d = np.pad(clip_p2d, ((0,0),(0,pad_len),(0,0),(0,0)), mode='constant', constant_values=0.0)

        clip_pose = poses_ds[st:ed]   # (tlen,24,3)
        if clip_pose.shape[0] < S:
            pad_len = S - clip_pose.shape[0]
            clip_pose = np.pad(clip_pose, ((0,pad_len),(0,0),(0,0)), mode='constant', constant_values=0.0)
            clip_trans = np.pad(trans_ds[st:ed], ((0,pad_len),(0,0)), mode='constant', constant_values=0.0)
        else:
            clip_trans = trans_ds[st:ed]

        # convert clip_pose axis-angle -> rotm -> 6d
        Rmats = axisangle_to_rotmat_batch(clip_pose)  # (S,24,3,3)
        sixd = rotmat_to_6d(Rmats)                   # (S,24,6)
        sixd_flat = sixd.reshape(S, -1)              # (S, 144)

        # compute foot contact: we don't have joint world positions directly here;
        # ideal: run SMPL forward kinematics using SMPL shape to get joint positions.
        # Here approximate foot contact by looking at root translation vertical velocity as cheap proxy.
        # For faithful reproduction, use AIST++ FK helpers (they provide smpl fits) - this is a placeholder.
        # We'll compute contacts from clip_trans vertical vel (cheap)
        jp = np.zeros((S,24,3), dtype=np.float32)  # placeholder joint pos (ALL zeros) -> contact zeros
        contacts = np.zeros((S,4), dtype=np.int8)
        # If you have joint positions (from SMPL forward kinematics), replace jp with those and call compute_foot_contact...
        # contacts = compute_foot_contact_from_joints(jp)

        # target motion: concat sixd_flat (144) + contacts (4) + root (3) -> 151
        root = clip_trans.reshape(S, 3)
        target_m = np.concatenate([sixd_flat, contacts.astype(np.float32), root.astype(np.float32)], axis=1)  # (S,151)

        # save as .npz: contains 'cond_2d' (views, S, 17, 3) and 'target_m' (S,151)
        out_name = f"{name}_clip{i:03d}.npz"
        np.savez_compressed(os.path.join(OUT_DIR, out_name), cond_2d=clip_p2d.astype(np.float32), target_m=target_m.astype(np.float32))
        count_saved += 1

print("Saved", count_saved, "ViMo-ready clips to", OUT_DIR)


  0%|          | 0/1408 [00:00<?, ?it/s]


ValueError: cannot reshape array of size 10800 into shape (6,24,3)

In [ ]:
!cd {str(API_DIR)} && cat requirements.txt

absl-py==0.9.0
numpy
torch
torchvision
opencv-python
git+https://github.com/liruilong940607/aniposelib
git+https://github.com/liruilong940607/smplx
ffmpeg-python
imageio
imageio-ffmpeg
gdown

In [ ]:
%%shell
pip install -q --upgrade pip
pip install -q absl-py
pip install -q numpy
pip install -q torch
pip install -q torchvision
pip install -q opencv-python
pip install -q git+https://github.com/liruilong940607/aniposelib
pip install -q git+https://github.com/liruilong940607/smplx
pip install -q ffmpeg-python
pip install -q imageio
pip install -q imageio-ffmpeg
pip install -q gdown

In [ ]:
print("Packages Installed:                      Version:\n\
---------------------------------------- -------------------")
!pip list | grep -E "^(absl-py|numpy|torch|torchvision|opencv-python|aniposelib|smplx|ffmpeg-python|imageio|imageio-ffmpeg|gdown)\s"

Package Installed:                       Version:
---------------------------------------- -------------------
absl-py                                  1.4.0
aniposelib                               0.3.9
ffmpeg-python                            0.2.0
gdown                                    5.2.0
imageio                                  2.37.0
imageio-ffmpeg                           0.6.0
numpy                                    2.0.2
opencv-python                            4.12.0.88
smplx                                    0.1.26
torch                                    2.8.0+cu126
torchvision                              0.23.0+cu126


In [ ]:
# Cell 1 - clone AlphaPose and install basic requirements
# NOTE: AlphaPose repo and exact install steps can change; this uses the official repo pattern.
git clone https://github.com/MVIG-SJTU/AlphaPose.git /content/AlphaPose
cd /content/AlphaPose
# install python deps (these are typical; Colab already has some)
pip install -r requirements.txt || true
# build if needed (AlphaPose sometimes needs cuda ops; this command is safe)
python setup.py build develop || true

echo "AlphaPose cloned to /content/AlphaPose. Follow AlphaPose README for any missing deps (detector weights)."


In [ ]:
ls -la /content/AlphaPose | head


In [ ]:
# Cell 2 - create input folder and place videos
mkdir -p /content/videos
# If you want a quick sample, uncomment and download a short clip:
# wget -O /content/videos/sample.mp4 "https://sample-videos.com/video123/mp4/240/big_buck_bunny_240p_5mb.mp4"
ls -la /content/videos


In [ ]:
# Cell 3 - run AlphaPose inference (example command)
# The exact demo script/arguments depend on the AlphaPose version; adapt as needed.
# This command attempts to run the 'inference' demo and produce JSON in COCO-like format.

cd /content/AlphaPose

# Example: run AlphaPose on a single video and output JSON (change paths to your video)
INPUT_VIDEO="/content/videos/sample.mp4"   # replace
OUTPUT_DIR="/content/alphapose_out"
mkdir -p "$OUTPUT_DIR"

# Common inference script (AlphaPose repo variants differ; this covers typical scripts)
# If your AlphaPose uses 'scripts/demo_inference.py' or 'alphapose.py', adapt the below.
# Try alphapose script (this is a common entrypoint)
python scripts/demo_inference.py \
    --cfg configs/coco/resnet/256x192_res50_lr1e-3_1x.yaml \
    --checkpoint checkpoints/res50_256x192.pth \
    --video "$INPUT_VIDEO" \
    --outdir "$OUTPUT_DIR" \
    --format json || echo "If this fails, check AlphaPose README for the demo entrypoint on this repo version."

echo "Look for JSON outputs in $OUTPUT_DIR (COCO-style per-frame keypoints & scores)."


In [ ]:
ls -la /content/alphapose_out | head


In [ ]:
# Cell 4 - Postprocess AlphaPose JSON outputs into cleaned numpy arrays following paper pipeline
# - Inputs: alphaPose JSON outputs (COCO-like)
# - Steps per paper: set coords/conf to zero if score<threshold; interpolation + zero-fill for missing joints.
# - Segment into windows of length S=150 (paper uses S=150).
import json, os, glob, numpy as np
from pathlib import Path
import math

ALPHAPOSE_OUT = "/content/alphapose_out"   # change if needed
S = 150
SCORE_THRESH = 0.3  # choose a threshold; paper uses "filter low-confidence joints" (use typical 0.3)
out_clips_dir = "/content/processed_2d"
os.makedirs(out_clips_dir, exist_ok=True)

def load_alphapose_json(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    # AlphaPose JSON format: list of dicts each with 'image_id', 'keypoints' flattened (51 values), 'score' etc.
    # We'll sort by image_id/frame index.
    data_sorted = sorted(data, key=lambda x: x.get('image_id', 0))
    frames = []
    for entry in data_sorted:
        kps = entry['keypoints']  # list length 51 -> x1,y1,conf1, x2,y2,conf2, ...
        arr = np.array(kps).reshape(17,3)  # 17 x 3
        frames.append(arr)
    return np.stack(frames)  # T x 17 x 3

def threshold_and_zero(frames, thresh=SCORE_THRESH):
    # frames: T x 17 x 3
    frames = frames.copy()
    mask = frames[:,:,2] < thresh  # low confidence
    frames[:,:,0][mask] = 0.0
    frames[:,:,1][mask] = 0.0
    frames[:,:,2][mask] = 0.0
    return frames

def interpolate_missing(frames):
    # frames: T x 17 x 3 -> do linear interpolation per joint coordinate separately
    T, J, _ = frames.shape
    out = frames.copy()
    for j in range(J):
        # x, y, conf as separate arrays
        for k in range(2):  # 0=x,1=y
            series = out[:, j, k]
            conf = out[:, j, 2]
            # treat conf==0 as missing -> set to nan for interpolation
            series_masked = series.copy().astype(float)
            series_masked[conf==0] = np.nan
            # simple linear interpolation: fill nan by linear interp; ends remain nan -> zero-fill
            nans = np.isnan(series_masked)
            if np.all(nans):
                series_masked[:] = 0.0
            else:
                # interp
                idx = np.arange(T)
                good = ~nans
                series_masked[nans] = np.interp(idx[nans], idx[good], series_masked[good])
            out[:, j, k] = series_masked
        # recompute confidence after interpolation: if originally all conf==0 -> remain zeros; else set conf to original conf or 0
        # Paper uses interpolation + zero-filling; keep confidences as original (0 if missing); do not invent confidences.
        # So keep out[:, j, 2] unchanged.
    # After interpolation, zero-fill any remaining nan (should be none)
    out = np.nan_to_num(out)
    return out

def segment_into_S(frames, S=S):
    T = frames.shape[0]
    clips = []
    if T < S:
        # pad to S with zeros (paper uses completion/interpolation; here zero-pad shorter clips)
        pad = np.zeros((S - T, frames.shape[1], frames.shape[2]), dtype=frames.dtype)
        frames_p = np.concatenate([frames, pad], axis=0)
        clips.append(frames_p)
    else:
        # sliding non-overlapping windows (or you can crop windows); paper uses S=150 training clips
        # We'll do non-overlapping chunks for simplicity
        num = T // S
        for i in range(num):
            clips.append(frames[i*S:(i+1)*S])
        rem = T % S
        if rem:
            # pad last partial window
            last = frames[num*S:]
            pad = np.zeros((S - rem, frames.shape[1], frames.shape[2]), dtype=frames.dtype)
            clips.append(np.concatenate([last, pad], axis=0))
    return clips

# Find JSON file(s) in output dir
json_files = glob.glob(os.path.join(ALPHAPOSE_OUT, "*.json"))
if len(json_files)==0:
    print("No AlphaPose JSON found in", ALPHAPOSE_OUT)
else:
    for jpath in json_files:
        print("Processing", jpath)
        frames = load_alphapose_json(jpath)            # T x 17 x 3
        th = threshold_and_zero(frames)
        interp = interpolate_missing(th)
        clips = segment_into_S(interp, S=S)
        base = Path(jpath).stem
        for idx, clip in enumerate(clips):
            outpath = os.path.join(out_clips_dir, f"{base}_clip{idx:03d}.npy")
            np.save(outpath, clip)   # saved: S x 17 x 3
        print(f"Saved {len(clips)} clips to {out_clips_dir}")

# Quick test: list one saved file and print shape
saved = glob.glob(os.path.join(out_clips_dir, "*.npy"))
if saved:
    a = np.load(saved[0])
    print("Example saved clip:", saved[0], "shape:", a.shape, "dtype:", a.dtype)
else:
    print("No processed clips found.")


In [ ]:
# Cell 5 - visualize first frame of a processed clip (simple matplotlib skeleton)
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path

processed = list(Path("/content/processed_2d").glob("*.npy"))
if not processed:
    print("No processed clips found; run previous cells.")
else:
    clip = np.load(processed[0])  # S x 17 x 3
    frame0 = clip[0]              # 17 x 3
    xs = frame0[:,0]; ys = frame0[:,1]; conf = frame0[:,2]

    # COCO skeleton edges (common order)
    skeleton = [
      (15,13),(13,11),(16,14),(14,12),(11,12),(5,11),
      (6,12),(5,6),(5,7),(6,8),(7,9),(8,10),(1,2),(0,1),(0,2),
      (1,3),(2,4),(3,5),(4,6)
    ]
    plt.figure(figsize=(4,6))
    plt.scatter(xs, -ys, c=conf+0.01, s=(conf*80)+5)  # flip y for visualization
    for a,b in skeleton:
        plt.plot([xs[a], xs[b]], [-ys[a], -ys[b]], linewidth=1)
    plt.title(processed[0].name + " first frame (flip y for visual)")
    plt.axis('off')
    plt.show()
